In [4]:
!pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached matplotlib-3.10.7-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.60.1-cp312-cp312-win_amd64.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp312-cp312-win_amd64.whl.metadata (6.4 kB)
  Using cached numpy-2.3.4-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached pillow-12.0.0-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.2.5-py3-none-any.whl.metadata (5.0 kB)
Using cached matplotlib-3.10.7-cp312-cp312-win_amd64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.60.1-cp312-cp312-win_amd64.whl (2.3 MB)
Using cached kiwisolver-1.4.9-cp312-cp312-win_amd64.whl (73 kB)
Using cached numpy-2.3.4-cp312-cp3


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\youss\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import serial as ser
import serial.tools.list_ports
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from collections import deque
import time

# Configuration
SERIAL_PORT = 'COM3'  # Change this to your Arduino port (COM3, COM4, etc. on Windows)
                       # On Mac: /dev/cu.usbmodem* or /dev/tty.usbmodem*
                       # On Linux: /dev/ttyACM0 or /dev/ttyUSB0
BAUD_RATE = 115200
WINDOW_SIZE = 500  # Number of data points to display

class PPGPlotter:
    def __init__(self, port, baud_rate, window_size):
        self.port = port
        self.baud_rate = baud_rate
        self.window_size = window_size
        
        # Data buffer
        self.data_buffer = deque(maxlen=window_size)
        self.time_buffer = deque(maxlen=window_size)
        
        # Initialize serial connection
        try:
            self.ser = ser.Serial(port, baud_rate, timeout=1)
            print(f"Connected to {port} at {baud_rate} baud")
            time.sleep(2)  # Wait for Arduino to reset
            
            # Clear initial startup messages
            for _ in range(20):
                try:
                    self.ser.readline()
                except:
                    pass
                    
        except ser.SerialException as e:
            print(f"Error opening serial port: {e}")
            print("\nAvailable ports:")
            ports = serial.tools.list_ports.comports()
            for p in ports:
                print(f"  {p.device} - {p.description}")
            exit(1)
        
        # Setup plot
        self.fig, self.ax = plt.subplots(figsize=(12, 6))
        self.line, = self.ax.plot([], [], 'r-', linewidth=1.5, label='Red LED PPG')
        
        self.ax.set_xlim(0, window_size)
        self.ax.set_ylim(0, 100000)  # Adjust based on your sensor values
        self.ax.set_xlabel('Sample Number', fontsize=12)
        self.ax.set_ylabel('PPG Signal Amplitude', fontsize=12)
        self.ax.set_title('MAX86916 PPG Signal - Live Plot', fontsize=14, fontweight='bold')
        self.ax.grid(True, alpha=0.3)
        self.ax.legend(loc='upper right')
        
        # Statistics text
        self.stats_text = self.ax.text(0.02, 0.98, '', transform=self.ax.transAxes,
                                       verticalalignment='top',
                                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        self.sample_count = 0
        self.start_time = time.time()
        
    def read_serial_data(self):
        """Read data from serial port"""
        try:
            if self.ser.in_waiting > 0:
                line = self.ser.readline().decode('utf-8', errors='ignore').strip()
                
                # Parse data in format "Red:12345"
                if line.startswith('Red:'):
                    value = int(line.split(':')[1])
                    self.data_buffer.append(value)
                    self.time_buffer.append(self.sample_count)
                    self.sample_count += 1
                    return True
        except Exception as e:
            print(f"Error reading serial: {e}")
        return False
    
    def update_plot(self, frame):
        """Update plot with new data"""
        # Read new data
        self.read_serial_data()
        
        if len(self.data_buffer) > 0:
            # Update line data
            self.line.set_data(list(self.time_buffer), list(self.data_buffer))
            
            # Auto-scale Y axis
            if len(self.data_buffer) > 10:
                min_val = min(self.data_buffer)
                max_val = max(self.data_buffer)
                margin = (max_val - min_val) * 0.1
                self.ax.set_ylim(min_val - margin, max_val + margin)
            
            # Update X axis to show latest data
            if self.sample_count > self.window_size:
                self.ax.set_xlim(self.sample_count - self.window_size, self.sample_count)
            
            # Update statistics
            elapsed = time.time() - self.start_time
            sample_rate = self.sample_count / elapsed if elapsed > 0 else 0
            current_val = self.data_buffer[-1]
            mean_val = sum(self.data_buffer) / len(self.data_buffer)
            
            stats = f'Samples: {self.sample_count}\n'
            stats += f'Rate: {sample_rate:.1f} Hz\n'
            stats += f'Current: {current_val}\n'
            stats += f'Mean: {mean_val:.0f}'
            self.stats_text.set_text(stats)
        
        return self.line, self.stats_text
    
    def start(self):
        """Start the animation"""
        print("Starting live plot... Place finger on sensor")
        print("Close the plot window to stop")
        
        ani = animation.FuncAnimation(
            self.fig, 
            self.update_plot, 
            interval=10,  # Update every 10ms
            blit=True,
            cache_frame_data=False
        )
        
        plt.tight_layout()
        plt.show()
        
        # Cleanup
        self.ser.close()
        print("\nSerial connection closed")

if __name__ == "__main__":
    print("=" * 50)
    print("MAX86916 PPG Live Plotter")
    print("=" * 50)
    print(f"\nAttempting to connect to {SERIAL_PORT}...")
    
    plotter = PPGPlotter(SERIAL_PORT, BAUD_RATE, WINDOW_SIZE)
    plotter.start()

ModuleNotFoundError: No module named 'serial.tools'